# Content-Based Filtering — Egyptian Real Estate Recommendation System

This notebook builds a **Content-Based Filtering** recommender system for the Egyptian real
estate dataset. Given a query property, the system returns the most similar properties based
on their intrinsic attributes: size, price, number of rooms, type, location, category, and
amenities.

**Approach overview**

| Step | Action |
|------|--------|
| 1 | Load dataset |
| 2 | Feature selection and engineering |
| 3 | Preprocessing via `ColumnTransformer` |
| 4 | Build combined feature matrix |
| 5 | Compute cosine similarity |
| 6 | Implement recommendation functions |
| 7 | Test and evaluate |
| 8 | Save artifacts |

**Constraints**
- Only `data/processed/cleaned_all_egypt.csv` is used (no synthetic data).
- No collaborative filtering, no clustering, no hybrid model.
- Memory-efficient similarity: uses a chunked top-K approach for ~63 k properties.


---
## 1. Import Required Libraries


In [ ]:
# Standard library
import ast
import os
import time
import warnings
from collections import Counter
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Scikit-learn
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Sparse matrices
import scipy.sparse as sp

# Persistence
import joblib

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 60)
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded successfully.')


---
## 2. Load the Dataset


In [ ]:
# Resolve paths relative to this notebook so it works regardless of
# where Jupyter was launched from.
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]   # notebooks/06_Recommendation/ -> project root
DATA_PATH    = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_all_egypt.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'models' / 'recommendation'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset      : {DATA_PATH}')
print(f'Artifact dir : {ARTIFACT_DIR}')


In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print(f'Dataset shape : {df_raw.shape}')
print(f'Columns       : {df_raw.columns.tolist()}')
df_raw.head(3)


In [ ]:
# Data audit: types, nulls, cardinality
audit = pd.DataFrame({
    'dtype'      : df_raw.dtypes,
    'null_count' : df_raw.isnull().sum(),
    'null_pct'   : (df_raw.isnull().mean() * 100).round(2),
    'unique'     : df_raw.nunique(),
})
print(audit.to_string())


---
## 3. Dataset Analysis and Feature Selection

For content-based filtering we need features that capture **what** a property is and
**where** it is located. The analysis below guides which columns to keep and how to
engineer them.

### 3.1 Categorical feature distributions


In [ ]:
cat_cols = ['category', 'property_type', 'price_period']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, cat_cols):
    counts = df_raw[col].value_counts()
    counts.plot(kind='bar', ax=ax, color=sns.color_palette('muted', len(counts)))
    ax.set_title(f'{col}  (n={df_raw[col].nunique()})', fontsize=12)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}',
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', fontsize=8)
plt.suptitle('Categorical Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### 3.2 Numerical feature distributions


In [ ]:
num_cols = ['price', 'size', 'bedrooms', 'bathrooms']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, num_cols):
    data = df_raw[col].dropna()
    q99  = data.quantile(0.99)
    ax.hist(data.clip(upper=q99), bins=40, edgecolor='white', color='steelblue')
    ax.set_title(f'{col} (clipped at 99th pct)', fontsize=11)
    ax.set_xlabel(col)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Numerical Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

df_raw[num_cols].describe().round(2)


### 3.3 Amenity frequency analysis


In [ ]:
def parse_list_column(series: pd.Series) -> pd.Series:
    """Safely parse a column whose values are string-encoded Python lists."""
    def _parse(val):
        try:
            result = ast.literal_eval(val)
            return result if isinstance(result, list) else []
        except (ValueError, SyntaxError):
            return []
    return series.apply(_parse)


# Human-readable labels for the 22 amenity codes
AMENITY_LABELS = {
    'AC': 'Air Conditioning', 'AN': 'Amenities Nearby', 'BA': 'Balcony',
    'BK': 'Basketball Court', 'BL': 'Built-in Kitchen', 'BW': 'Built-in Wardrobes',
    'CO': 'Covered Parking',  'CP': 'Central Park',     'CR': 'Concierge',
    'DN': 'Dining Area',      'LB': 'Lobby',            'MR': 'Maid Room',
    'PG': 'Playground',       'PN': 'Penthouse',        'PP': 'Private Pool',
    'SE': 'Security',         'SP': 'Shared Pool',      'SS': 'Shared Spa',
    'ST': 'Storage',          'SY': 'Study Room',       'VW': 'View',
    'WC': 'Gym'
}

amenity_lists_raw = parse_list_column(df_raw['amenities'])
all_codes         = [code for sublist in amenity_lists_raw for code in sublist]
amen_freq         = Counter(all_codes)
amen_df           = pd.DataFrame(amen_freq.most_common(), columns=['Code', 'Count'])
amen_df['Label']  = amen_df['Code'].map(AMENITY_LABELS)

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(amen_df['Label'], amen_df['Count'], color=sns.color_palette('muted', len(amen_df)))
ax.set_title('Amenity Frequency across All Properties', fontsize=13)
ax.set_xlabel('Amenity')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

print(amen_df.to_string(index=False))


### 3.4 Feature selection rationale

| Feature | Type | Rationale | Action |
|---------|------|-----------|--------|
| `price` | Numerical | Core search criterion | Keep; log-transform to compress skew |
| `size` | Numerical | Primary physical descriptor | Keep; log-transform |
| `bedrooms` | Numerical | Key lifestyle metric | Keep; median-fill missing |
| `bathrooms` | Numerical | Key lifestyle metric | Keep; median-fill missing |
| `latitude` | Numerical | Geographic position | Keep |
| `longitude` | Numerical | Geographic position | Keep |
| `category` | Categorical | rent vs. buy vs. commercial | Keep; OneHotEncoder |
| `property_type` | Categorical | Villa, Apartment, Office, etc. | Keep; OneHotEncoder |
| `city` | Categorical (derived) | Top-level location signal | Extract from `location`; OneHotEncoder |
| `district` | Categorical (derived) | Mid-level location signal | Extract from `location`; OneHotEncoder |
| `amenities` | Multi-label | Binary presence of 22 amenity codes | MultiLabelBinarizer |
| `id`, `title`, `images`, `share_url`, `currency`, `price_raw_value`, `price_period`, `price_type` | Meta/redundant | No similarity signal | Excluded |


---
## 4. Preprocessing

### 4.1 Feature engineering and missing value handling


In [ ]:
def extract_location_levels(location_series: pd.Series):
    """
    Split the comma-separated location string into (city, district).

    Convention: the last segment is the city, the second-to-last is the district.

    Parameters
    ----------
    location_series : pd.Series of strings

    Returns
    -------
    (city: pd.Series, district: pd.Series)
    """
    parts    = location_series.str.split(',').apply(lambda x: [p.strip() for p in x])
    city     = parts.apply(lambda x: x[-1])
    district = parts.apply(lambda x: x[-2] if len(x) >= 2 else x[-1])
    return city, district


def build_feature_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a cleaned feature DataFrame ready for the ColumnTransformer.

    - Applies log1p to price and size to reduce right skew.
    - Fills missing bedrooms/bathrooms with the column median.
    - Extracts city and district from the location string.

    The original DataFrame index is preserved.
    """
    feat = pd.DataFrame(index=df.index)

    # Numerical
    feat['log_price']  = np.log1p(df['price'])
    feat['log_size']   = np.log1p(df['size'])
    feat['bedrooms']   = df['bedrooms'].fillna(df['bedrooms'].median())
    feat['bathrooms']  = df['bathrooms'].fillna(df['bathrooms'].median())
    feat['latitude']   = df['latitude']
    feat['longitude']  = df['longitude']

    # Categorical
    feat['category']      = df['category'].fillna('unknown')
    feat['property_type'] = df['property_type'].fillna('unknown')

    city, district        = extract_location_levels(df['location'])
    feat['city']          = city
    feat['district']      = district

    return feat


df            = df_raw.copy()
features_df   = build_feature_dataframe(df)
amenity_lists = parse_list_column(df['amenities'])

print('Feature DataFrame shape:', features_df.shape)
print('Null counts after engineering:')
print(features_df.isnull().sum())
features_df.head(3)


### 4.2 ColumnTransformer for numerical and categorical features


In [ ]:
NUMERICAL_FEATURES   = ['log_price', 'log_size', 'bedrooms', 'bathrooms',
                        'latitude', 'longitude']

CATEGORICAL_FEATURES = ['category', 'property_type', 'city', 'district']

numerical_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline,   NUMERICAL_FEATURES),
        ('cat', categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder='drop'
)

print('ColumnTransformer configured.')
print('  Numerical  :', NUMERICAL_FEATURES)
print('  Categorical:', CATEGORICAL_FEATURES)


### 4.3 MultiLabelBinarizer for amenities

Each property has a variable-length list of amenity codes.  
MultiLabelBinarizer converts that into a binary matrix of shape *(n_properties, 22)*.


In [ ]:
mlb                  = MultiLabelBinarizer()
amenity_matrix_dense = mlb.fit_transform(amenity_lists)
amenity_matrix       = sp.csr_matrix(amenity_matrix_dense)

print(f'Amenity codes ({len(mlb.classes_)}): {mlb.classes_.tolist()}')
print(f'Amenity matrix shape: {amenity_matrix.shape}')


---
## 5. Build the Combined Feature Matrix

Three blocks are concatenated horizontally into one sparse CSR matrix:

```
feature_matrix = [ num_block | cat_block | amenity_block ]
```

Using a sparse representation avoids the ~several-GB memory cost of a dense 63 k x N matrix.


In [ ]:
# Fit and transform
structured_matrix = preprocessor.fit_transform(features_df)   # already sparse (CSR)

# Horizontal stack
feature_matrix = sp.hstack([structured_matrix, amenity_matrix], format='csr')

n_props, n_feats = feature_matrix.shape
mem_mb = (feature_matrix.data.nbytes + feature_matrix.indices.nbytes +
          feature_matrix.indptr.nbytes) / 1024 ** 2

print(f'Combined feature matrix : {n_props:,} properties x {n_feats} features')
print(f'Sparse memory usage     : {mem_mb:.1f} MB')
print(f'Sparsity                : {1 - feature_matrix.nnz / (n_props * n_feats):.2%}')


### 5.1 Feature name reference


In [ ]:
def get_feature_names(ct: ColumnTransformer, mlb_obj: MultiLabelBinarizer) -> list:
    """Return ordered feature names matching columns of the combined feature matrix."""
    names = []
    for name, pipe, cols in ct.transformers_:
        if name == 'num':
            names.extend(cols)
        elif name == 'cat':
            ohe = pipe.named_steps['encoder']
            names.extend(ohe.get_feature_names_out(cols).tolist())
    names.extend([f'amenity_{c}' for c in mlb_obj.classes_])
    return names


feature_names = get_feature_names(preprocessor, mlb)
print(f'Total features : {len(feature_names)}')
print('First 20       :', feature_names[:20])
print('Last  10       :', feature_names[-10:])


---
## 6. Compute Property-to-Property Cosine Similarity

### Memory-efficient strategy

Computing the full N x N similarity matrix for 63 k properties would require  
63 000^2 x 4 bytes ~ **15 GB of RAM**.

Instead we use a **chunked top-K pre-indexing** strategy:
- Process rows in chunks of `CHUNK_SIZE`.
- For each chunk, compute cosine similarity against the full matrix.
- Keep only the top `TOP_K` neighbors per property.
- Store two compact NumPy arrays: `top_indices` and `top_scores`.

This yields instant lookup at inference time and uses only megabytes of RAM.


In [ ]:
# Property ID <-> row-index mapping
property_ids = df['id'].values
id_to_idx    = {pid: i for i, pid in enumerate(property_ids)}
idx_to_id    = {i: pid for i, pid in enumerate(property_ids)}

print(f'ID mapping created for {len(id_to_idx):,} properties.')
print('Sample id_to_idx:', list(id_to_idx.items())[:5])


In [ ]:
TOP_K      = 50     # neighbors stored per property
CHUNK_SIZE = 1000   # rows processed at once (tune for available RAM)

n_total = feature_matrix.shape[0]

# Pre-allocate output arrays
top_indices = np.empty((n_total, TOP_K), dtype=np.int32)
top_scores  = np.empty((n_total, TOP_K), dtype=np.float32)

t0 = time.time()

for start in range(0, n_total, CHUNK_SIZE):
    end       = min(start + CHUNK_SIZE, n_total)
    chunk     = feature_matrix[start:end]

    # sim_block shape: (chunk_size, n_total)
    sim_block = cosine_similarity(chunk, feature_matrix)

    for local_i, global_i in enumerate(range(start, end)):
        row      = sim_block[local_i]

        # O(n) retrieval of top K+1 candidates, then exclude the property itself
        part_idx = np.argpartition(row, -(TOP_K + 1))[-(TOP_K + 1):]
        part_idx = part_idx[part_idx != global_i]

        # Sort by descending score and keep top K
        sort_idx = part_idx[np.argsort(row[part_idx])[::-1]][:TOP_K]

        top_indices[global_i] = sort_idx
        top_scores[global_i]  = row[sort_idx].astype(np.float32)

    if (start // CHUNK_SIZE) % 10 == 0:
        elapsed = time.time() - t0
        pct     = (end / n_total) * 100
        print(f'  {pct:5.1f}%  ({end:,}/{n_total:,})   elapsed: {elapsed:.0f}s')

elapsed    = time.time() - t0
mem_idx_mb = (top_indices.nbytes + top_scores.nbytes) / 1024 ** 2
print(f'\nDone.  Total time: {elapsed:.0f}s   Index memory: {mem_idx_mb:.1f} MB')


---
## 7. Recommendation Functions

Two reusable public functions are provided:
- `get_property_information()` — retrieve a human-readable summary of a property by ID.
- `recommend_similar_properties()` — return the top-N most similar properties for a given ID.


In [ ]:
def get_property_information(
        property_id: int,
        dataframe: pd.DataFrame,
        id_map: dict
) -> pd.Series:
    """
    Retrieve a single property record by its original ID.

    Parameters
    ----------
    property_id : int
        The property's original `id` value in the dataset.
    dataframe : pd.DataFrame
        The full dataset.
    id_map : dict
        Mapping property_id -> row index.

    Returns
    -------
    pd.Series with selected display columns.

    Raises
    ------
    KeyError if property_id is not found.
    """
    if property_id not in id_map:
        raise KeyError(f'Property ID {property_id} not found in the dataset.')

    idx    = id_map[property_id]
    record = dataframe.iloc[idx]

    display_cols = ['id', 'title', 'price', 'category', 'property_type',
                    'bedrooms', 'bathrooms', 'size', 'location', 'amenities']
    return record[display_cols]


# ---------------------------------------------------------------------------


def recommend_similar_properties(
        property_id: int,
        dataframe: pd.DataFrame,
        id_map: dict,
        idx_map: dict,
        top_idx_array: np.ndarray,
        top_score_array: np.ndarray,
        n_recommendations: int = 10
) -> pd.DataFrame:
    """
    Return the top-N most similar properties for a given property ID.

    Parameters
    ----------
    property_id       : int  -- query property's original ID
    dataframe         : full dataset
    id_map            : property_id -> row index
    idx_map           : row index -> property_id
    top_idx_array     : shape (n_props, TOP_K) -- pre-computed neighbor indices
    top_score_array   : shape (n_props, TOP_K) -- corresponding similarity scores
    n_recommendations : number of results to return (must be <= TOP_K)

    Returns
    -------
    pd.DataFrame with columns:
        rank, similarity_score, id, title, price, category,
        property_type, bedrooms, bathrooms, size, location, amenities
    """
    if property_id not in id_map:
        raise KeyError(f'Property ID {property_id} not found in the dataset.')

    n_recommendations = min(n_recommendations, top_idx_array.shape[1])

    query_idx     = id_map[property_id]
    neighbor_idxs = top_idx_array[query_idx,  :n_recommendations]
    neighbor_sims = top_score_array[query_idx, :n_recommendations]

    display_cols = ['id', 'title', 'price', 'category', 'property_type',
                    'bedrooms', 'bathrooms', 'size', 'location', 'amenities']

    results = dataframe.iloc[neighbor_idxs][display_cols].copy()
    results.insert(0, 'similarity_score', np.round(neighbor_sims.astype(float), 4))
    results.insert(0, 'rank', range(1, n_recommendations + 1))
    results.reset_index(drop=True, inplace=True)

    return results


print('Recommendation functions defined.')


---
## 8. Testing the Recommender

We test with **five different property IDs** covering various types and locations.


In [ ]:
# Sample one property per category, plus one Villa for diversity
sample_ids = (
    df.groupby('category')
      .apply(lambda g: g.sample(1, random_state=42))
      .reset_index(drop=True)['id']
      .tolist()
)
extra_villa = df[df['property_type'] == 'Villa'].sample(1, random_state=7)['id'].values[0]
TEST_IDS    = sample_ids + [extra_villa]

print('Test property IDs:', TEST_IDS)


In [ ]:
for pid in TEST_IDS:
    print('=' * 80)
    print(f'QUERY PROPERTY  (id={pid})')
    print('=' * 80)

    info = get_property_information(pid, df, id_to_idx)
    for field, val in info.items():
        print(f'  {field:<18}: {val}')

    recs = recommend_similar_properties(
        pid, df, id_to_idx, idx_to_id,
        top_indices, top_scores, n_recommendations=5
    )
    print()
    print('TOP-5 SIMILAR PROPERTIES')
    print(recs[['rank', 'similarity_score', 'id', 'property_type',
                'price', 'size', 'bedrooms', 'bathrooms', 'category']].to_string(index=False))
    print()


---
## 9. Evaluation

For a content-based system without explicit user feedback there is no ground-truth relevance
label. We evaluate **intrinsic quality** along three axes:

1. **Similarity score distribution** — are scores high (meaningful neighbors) or low (noise)?
2. **Feature overlap** — how many key attributes do query and neighbors share?
3. **Aggregated match rates** — consistency across different query property types.

### 9.1 Similarity score distribution per query


In [ ]:
fig, axes = plt.subplots(1, len(TEST_IDS), figsize=(4 * len(TEST_IDS), 4), sharey=False)

for ax, pid in zip(axes, TEST_IDS):
    idx    = id_to_idx[pid]
    scores = top_scores[idx, :10].astype(float)
    ptype  = df.loc[df['id'] == pid, 'property_type'].values[0]
    ax.bar(range(1, 11), scores, color='steelblue', edgecolor='white')
    ax.set_title(f'ID {pid}\n{ptype}', fontsize=9)
    ax.set_xlabel('Neighbor rank')
    ax.set_ylabel('Cosine similarity')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(range(1, 11))

plt.suptitle('Top-10 Similarity Scores per Query Property', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### 9.2 Shared feature analysis


In [ ]:
def shared_features_summary(
        query_id: int,
        recommendations: pd.DataFrame,
        dataframe: pd.DataFrame,
        id_map: dict
) -> pd.DataFrame:
    """
    For each recommended property, report whether key categorical features
    match the query property.

    Checked features: category, property_type, city, district.

    Returns
    -------
    pd.DataFrame with binary match columns and a total_matches summary column.
    """
    q_idx          = id_map[query_id]
    q_row          = dataframe.iloc[q_idx]
    city_q, dist_q = extract_location_levels(pd.Series([q_row['location']]))

    rows = []
    for _, rec in recommendations.iterrows():
        rec_row        = dataframe.loc[dataframe['id'] == rec['id']].iloc[0]
        city_r, dist_r = extract_location_levels(pd.Series([rec_row['location']]))
        match = {
            'rank'             : rec['rank'],
            'id'               : rec['id'],
            'similarity_score' : rec['similarity_score'],
            'category_match'   : int(rec_row['category']      == q_row['category']),
            'type_match'       : int(rec_row['property_type'] == q_row['property_type']),
            'city_match'       : int(city_r.values[0]         == city_q.values[0]),
            'district_match'   : int(dist_r.values[0]         == dist_q.values[0]),
        }
        match['total_matches'] = sum([
            match['category_match'], match['type_match'],
            match['city_match'],     match['district_match']
        ])
        rows.append(match)

    return pd.DataFrame(rows)


# Demonstrate for the first test property
pid0   = TEST_IDS[0]
recs0  = recommend_similar_properties(
    pid0, df, id_to_idx, idx_to_id,
    top_indices, top_scores, n_recommendations=10
)
shared = shared_features_summary(pid0, recs0, df, id_to_idx)

print(f'Shared feature analysis for query ID: {pid0}')
print(shared.to_string(index=False))


### 9.3 Aggregated evaluation across all test queries


In [ ]:
agg_rows = []
for pid in TEST_IDS:
    recs = recommend_similar_properties(
        pid, df, id_to_idx, idx_to_id,
        top_indices, top_scores, n_recommendations=10
    )
    sh = shared_features_summary(pid, recs, df, id_to_idx)
    agg_rows.append({
        'property_id'        : pid,
        'property_type'      : df.loc[df['id'] == pid, 'property_type'].values[0],
        'avg_similarity'     : recs['similarity_score'].mean(),
        'pct_category_match' : sh['category_match'].mean() * 100,
        'pct_type_match'     : sh['type_match'].mean()     * 100,
        'pct_city_match'     : sh['city_match'].mean()     * 100,
        'pct_district_match' : sh['district_match'].mean() * 100,
    })

agg_df = pd.DataFrame(agg_rows)
print('Aggregated evaluation (top-10 neighbors per query):')
print(agg_df.round(2).to_string(index=False))


### 9.4 Visual evaluation summary


In [ ]:
match_cols = ['pct_category_match', 'pct_type_match', 'pct_city_match', 'pct_district_match']
labels     = ['Category', 'Property Type', 'City', 'District']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: mean feature match rates
mean_matches = agg_df[match_cols].mean()
axes[0].bar(labels, mean_matches.values, color=sns.color_palette('muted', 4))
axes[0].set_ylim(0, 110)
axes[0].set_ylabel('% of top-10 neighbors matching')
axes[0].set_title('Average Feature Match Rate\n(across all test queries)', fontsize=11)
for i, v in enumerate(mean_matches.values):
    axes[0].text(i, v + 1.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Right: mean similarity score per query type
x_ticks = range(len(agg_df))
axes[1].bar(x_ticks, agg_df['avg_similarity'],
            color=sns.color_palette('muted', len(agg_df)))
axes[1].set_xticks(x_ticks)
axes[1].set_xticklabels(agg_df['property_type'], rotation=30, ha='right')
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Mean cosine similarity (top-10)')
axes[1].set_title('Mean Similarity Score per Query', fontsize=11)
for i, row in agg_df.iterrows():
    axes[1].text(i, row['avg_similarity'] + 0.01,
                 f"{row['avg_similarity']:.3f}",
                 ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Content-Based Recommender — Intrinsic Evaluation', fontsize=13)
plt.tight_layout()
plt.show()


### 9.5 Example recommendation display


In [ ]:
DEMO_ID = TEST_IDS[1]

print('QUERY PROPERTY')
print('-' * 60)
for k, v in get_property_information(DEMO_ID, df, id_to_idx).items():
    print(f'  {k:<18}: {v}')

print()
print('TOP-10 RECOMMENDATIONS')
print('-' * 60)
demo_recs = recommend_similar_properties(
    DEMO_ID, df, id_to_idx, idx_to_id,
    top_indices, top_scores, n_recommendations=10
)
demo_recs


---
## 10. Save Artifacts

All objects required by downstream notebooks are persisted to `models/recommendation/`.

| File | Contents |
|------|----------|
| `preprocessor.joblib` | Fitted `ColumnTransformer` (StandardScaler + OneHotEncoder) |
| `mlb.joblib` | Fitted `MultiLabelBinarizer` for amenities |
| `feature_matrix.npz` | Combined sparse feature matrix (n x d) |
| `top_indices.npy` | Top-K neighbor row indices per property |
| `top_scores.npy` | Corresponding cosine similarity scores |
| `id_to_idx.joblib` | Dict: property_id -> row index |
| `idx_to_id.joblib` | Dict: row index -> property_id |
| `feature_names.joblib` | Ordered list of feature names |
| `amenity_labels.joblib` | Dict: amenity code -> human-readable label |


In [ ]:
def save_artifact(obj, filename: str, artifact_dir: Path, use_numpy: bool = False) -> None:
    """
    Save a Python object to disk and print a confirmation line.

    Parameters
    ----------
    obj          : object to persist
    filename     : output filename (relative to artifact_dir)
    artifact_dir : target directory
    use_numpy    : if True, save with np.save; otherwise use joblib.dump
    """
    path = artifact_dir / filename
    if use_numpy:
        np.save(path, obj)
    else:
        joblib.dump(obj, path)
    size_mb = path.stat().st_size / 1024 ** 2
    print(f'  Saved {filename:<42}  ({size_mb:.2f} MB)')


print(f'Saving artifacts to: {ARTIFACT_DIR}')
print()

save_artifact(preprocessor,   'preprocessor.joblib',  ARTIFACT_DIR)
save_artifact(mlb,            'mlb.joblib',           ARTIFACT_DIR)

# Sparse matrix requires scipy's own serializer
sp.save_npz(ARTIFACT_DIR / 'feature_matrix.npz', feature_matrix)
fm_mb = (ARTIFACT_DIR / 'feature_matrix.npz').stat().st_size / 1024 ** 2
print(f'  Saved feature_matrix.npz                           ({fm_mb:.2f} MB)')

save_artifact(top_indices,    'top_indices.npy',      ARTIFACT_DIR, use_numpy=True)
save_artifact(top_scores,     'top_scores.npy',       ARTIFACT_DIR, use_numpy=True)
save_artifact(id_to_idx,      'id_to_idx.joblib',     ARTIFACT_DIR)
save_artifact(idx_to_id,      'idx_to_id.joblib',     ARTIFACT_DIR)
save_artifact(feature_names,  'feature_names.joblib', ARTIFACT_DIR)
save_artifact(AMENITY_LABELS, 'amenity_labels.joblib',ARTIFACT_DIR)

print()
print('All artifacts saved successfully.')


In [ ]:
# Verification: reload every artifact and run a live recommendation query
loaded_preprocessor   = joblib.load(ARTIFACT_DIR / 'preprocessor.joblib')
loaded_mlb            = joblib.load(ARTIFACT_DIR / 'mlb.joblib')
loaded_feature_matrix = sp.load_npz(ARTIFACT_DIR / 'feature_matrix.npz')
loaded_top_indices    = np.load(ARTIFACT_DIR / 'top_indices.npy')
loaded_top_scores     = np.load(ARTIFACT_DIR / 'top_scores.npy')
loaded_id_to_idx      = joblib.load(ARTIFACT_DIR / 'id_to_idx.joblib')
loaded_idx_to_id      = joblib.load(ARTIFACT_DIR / 'idx_to_id.joblib')

# Sanity check with a live query
test_recs = recommend_similar_properties(
    TEST_IDS[0], df, loaded_id_to_idx, loaded_idx_to_id,
    loaded_top_indices, loaded_top_scores, n_recommendations=3
)

print('Artifact reload verification passed.')
print(f'Feature matrix : {loaded_feature_matrix.shape}')
print(f'Top-indices    : {loaded_top_indices.shape}')
print(f'Top-scores     : {loaded_top_scores.shape}')
print()
print('Sample recommendations from reloaded artifacts:')
test_recs[['rank', 'similarity_score', 'id', 'property_type', 'price', 'category']]


---
## Summary

| Component | Details |
|-----------|----------|
| Dataset | `cleaned_all_egypt.csv` — 62,977 properties, 18 columns |
| Features used | 6 numerical + 4 categorical + 22 amenity binary flags |
| Numerical preprocessing | `StandardScaler` (log-transformed price and size) |
| Categorical preprocessing | `OneHotEncoder` (category, property_type, city, district) |
| Amenity preprocessing | `MultiLabelBinarizer` |
| Feature matrix | Sparse CSR — 62,977 x N features |
| Similarity metric | Cosine similarity |
| Index | Top-50 neighbors per property, pre-computed with chunked processing |
| Artifacts saved | 9 files in `models/recommendation/` |

The model is ready. All saved artifacts can be loaded directly in subsequent notebooks
for hybrid recommendation, evaluation, or deployment.
